In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
# Install required packages
!pip install scanpy scvi-tools seaborn torch anndata scikit-misc numpy


  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 97.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.0/183.0 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.9/151.9 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 kB 4.6 MB/s eta 0:00:00
 

In [3]:
import os
import tempfile
import numpy as np
import scanpy as sc
import scvi
import seaborn as sns
import torch
import anndata as ad

In [4]:
# -------------------------
# Setup
# -------------------------
scvi.settings.seed = 0
print("Last run with scvi-tools version:", scvi.__version__)

sc.set_figure_params(figsize=(6, 6), frameon=False)
sns.set_theme()
torch.set_float32_matmul_precision("high")

save_dir = "/content/drive/MyDrive/Colab Notebooks/scvi_test/scvi_embeddings"
os.makedirs(save_dir, exist_ok=True)


INFO: Seed set to 0
INFO:lightning.fabric.utilities.seed:Seed set to 0


Last run with scvi-tools version: 1.4.2


In [5]:

# -------------------------
# Data Loading & Prep (Done ONCE)
# -------------------------
print("\n===== Preparing Biddy (CellTag) Data =====")
adata = ad.read_h5ad('/content/drive/MyDrive/Colab Notebooks/data/Biddy_scvi_train.h5ad')
print("Original Train shape:", adata.shape)

adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.raw = adata

# Calculate and subset to 2000 HVGs
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=2000,
    subset=True,
    layer="counts",
    flavor="seurat_v3",
    batch_key=None,
)
print("Filtered Train shape:", adata.shape)

# Setup train AnnData for scVI
scvi.model.SCVI.setup_anndata(
    adata,
    layer="counts",
)

# Load Test Data
test_adata = ad.read_h5ad('/content/drive/MyDrive/Colab Notebooks/data/Biddy_scvi_test.h5ad')
print("Original Test shape:", test_adata.shape)

# CRITICAL FIX: Slice test data to match the exact 2000 genes from train data
# (If your test data is ALREADY subsetted to match these exact 2000 genes,
# you can comment out the next line)
test_adata = test_adata[:, adata.var_names].copy()

test_adata.layers["counts"] = test_adata.X.copy()
print("Filtered Test shape:", test_adata.shape)




===== Preparing Biddy (CellTag) Data =====
Original Train shape: (5893, 2000)
Filtered Train shape: (5893, 2000)
Original Test shape: (641, 2000)
Filtered Test shape: (641, 2000)


In [6]:
# -------------------------
# Run scVI Sweep
# -------------------------
latent_dims = [10, 32]

for n_latent in latent_dims:
    print(f"\n===== Running scVI with n_latent = {n_latent} =====")

    # Initialize and Train model
    model = scvi.model.SCVI(
        adata,
        n_latent=n_latent
    )
    model.train()

    # Save train embedding
    train_latent = model.get_latent_representation()
    train_filename = f'Biddy_scvi_train_latent{n_latent}.npy'
    np.save(os.path.join(save_dir, train_filename), train_latent)
    print(f"Saved {train_filename}:", train_latent.shape)

    # Save test embedding
    test_latent = model.get_latent_representation(test_adata)
    test_filename = f'Biddy_scvi_test_latent{n_latent}.npy'
    np.save(os.path.join(save_dir, test_filename), test_latent)
    print(f"Saved {test_filename}:", test_latent.shape)

print("\nAll scVI runs for Biddy (CellTag) are complete!")

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.



===== Running scVI with n_latent = 10 =====


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Training:   0%|          | 0/400 [00:00<?, ?it/s]

INFO: `Trainer.fit` stopped: `max_epochs=400` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=400` reached.


...

Saved Biddy_scvi_test_latent10.npy: (641, 10)

===== Running scVI with n_latent = 32 =====


/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Training:   0%|          | 0/400 [00:00<?, ?it/s]

INFO: `Trainer.fit` stopped: `max_epochs=400` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=400` reached.


Saved Biddy_scvi_train_latent32.npy: (5893, 32)
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             
Saved Biddy_scvi_test_latent32.npy: (641, 32)

All scVI runs for Biddy (CellTag) are complete!
